# Day 5 – Supervised Learning


---

## 1. Learning Objectives

- Distinguish classification from regression problems.
- Build a student performance predictor (classification).
- Build a house price predictor (regression).
- Build a spam detector (text classification with TF-IDF).
- Apply disease detection (multi-feature classification).
- Select and interpret appropriate evaluation metrics for each problem type.

## 2. Day Overview

**Total Learning Time: 5 Hours**

| Block | Time | Topic |
|---|---|---|
| Theory | 45 min | Supervised learning types & metrics |
| Lab 1 | 30 min | Student performance — classification |
| Lab 2 | 30 min | House price — regression |
| Lab 3 | 30 min | Spam detection — text classification |
| Lab 4 | 30 min | Disease detection — synthetic |
| Lab 5 | 30 min | Model comparison: class vs regression metrics |
| Exercises | 30 min | Practice |
| Review | 15 min | Summary & Knowledge Check |

## 3. Concept / Theory

### Classification vs Regression

| Property | Classification | Regression |
|---|---|---|
| Target (y) | Discrete categories | Continuous numbers |
| Output | Class label | Numeric value |
| Key metrics | Accuracy, F1, AUC | MAE, MSE, R² |
| Example | Spam or not? | House price? |

### Regression Metrics
- **MAE (Mean Absolute Error):** average absolute difference
- **MSE (Mean Squared Error):** penalizes large errors
- **RMSE:** √MSE, same units as y
- **R²:** fraction of variance explained (1.0 = perfect)

### Classification Metrics (recap)
- **Accuracy** = correct / total
- **Precision** = TP / (TP+FP)
- **Recall** = TP / (TP+FN)
- **F1** = harmonic mean(Precision, Recall)

In [ ]:
# Setup
!pip install -q scikit-learn pandas numpy matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, classification_report,
                              mean_absolute_error, mean_squared_error, r2_score)
print('Ready!')

## 4. Practical Lab 1 – Student Performance Prediction (Classification)

### Objective
Predict Pass/Fail from study hours, attendance, and previous score.

### Explanation
This is a **binary classification** problem. We use Logistic Regression as a baseline.

In [ ]:
# Lab 1: Generate synthetic student data
np.random.seed(42)
n = 300
df = pd.DataFrame({
    'Hours_Studied':  np.random.uniform(1, 10, n),
    'Attendance_Pct': np.random.uniform(50, 100, n),
    'Prev_Score':     np.random.uniform(35, 90, n)
})
# Create label: students who study more, attend more, and scored higher tend to pass
signal = (0.5*df['Hours_Studied'] + 0.04*df['Attendance_Pct'] + 0.05*df['Prev_Score'])
p = 1 / (1 + np.exp(-(signal - 6.5)))
df['Result'] = np.where(np.random.rand(n) < p, 'Pass', 'Fail')

print('Dataset preview:')
print(df.head())
print('\nClass distribution:')
print(df['Result'].value_counts())

In [ ]:
# Train Logistic Regression
X = df[['Hours_Studied','Attendance_Pct','Prev_Score']]
y = df['Result']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

clf = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(random_state=42))])
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

print('Accuracy:', accuracy_score(y_te, y_pred))
print(classification_report(y_te, y_pred))

# Predict new student
new = pd.DataFrame({'Hours_Studied':[7.5], 'Attendance_Pct':[85], 'Prev_Score':[72]})
print('\nNew student prediction:', clf.predict(new)[0])
print('Probability (Fail/Pass):', clf.predict_proba(new)[0].round(3))

### Expected Output
```
Accuracy: ~0.85
New student prediction: Pass
```

## 5. Practical Lab 2 – House Price Prediction (Regression)

### Objective
Predict house prices from features like size, rooms, and age.

### Explanation
Regression predicts continuous values. We use Ridge Regression (L2 regularization) which prevents overfitting.

In [ ]:
# Lab 2: Generate synthetic housing data
np.random.seed(42)
n = 400
df_h = pd.DataFrame({
    'Size_sqft':   np.random.uniform(500, 3000, n),
    'Bedrooms':    np.random.randint(1, 6, n),
    'Age_years':   np.random.randint(1, 50, n),
    'Distance_km': np.random.uniform(1, 30, n)
})
# Price formula
df_h['Price'] = (
    150 * df_h['Size_sqft'] +
    25000 * df_h['Bedrooms'] -
    500 * df_h['Age_years'] -
    3000 * df_h['Distance_km'] +
    np.random.randn(n) * 15000 + 50000
)
print('Housing dataset preview:')
print(df_h.describe().round(0))

In [ ]:
# Train Ridge Regression
Xh = df_h[['Size_sqft','Bedrooms','Age_years','Distance_km']]
yh = df_h['Price']

Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(Xh, yh, test_size=0.2, random_state=42)

reg = Pipeline([('sc', StandardScaler()), ('ridge', Ridge(alpha=1.0))])
reg.fit(Xh_tr, yh_tr)
yh_pred = reg.predict(Xh_te)

mae  = mean_absolute_error(yh_te, yh_pred)
rmse = mean_squared_error(yh_te, yh_pred) ** 0.5
r2   = r2_score(yh_te, yh_pred)
print(f'MAE:  {mae:,.0f}')
print(f'RMSE: {rmse:,.0f}')
print(f'R²:   {r2:.4f}')

# Visualize predictions vs actual
plt.figure(figsize=(7, 5))
plt.scatter(yh_te, yh_pred, alpha=0.5)
plt.plot([yh_te.min(), yh_te.max()], [yh_te.min(), yh_te.max()], 'r--')
plt.xlabel('Actual Price'); plt.ylabel('Predicted Price')
plt.title('Actual vs Predicted House Price')
plt.tight_layout(); plt.show()

### Expected Output
```
R²: ~0.97  (model explains 97% of variance)
```

## 6. Practical Lab 3 – Spam Email Detection (Text Classification)

### Objective
Build a text classifier using TF-IDF features.

### Explanation
**TF-IDF** (Term Frequency-Inverse Document Frequency) converts raw text into numeric feature vectors. Logistic Regression then classifies these vectors.

In [ ]:
# Lab 3: Synthetic spam dataset
spam_emails = [
    'Win a free iPhone now click here',
    'Congratulations you have won lottery click to claim',
    'Earn money fast online work from home',
    'Free credit card approval guaranteed cash',
    'Limited time offer buy now get 50% off',
    'Buy cheap pills online no prescription needed',
    'Meeting tomorrow at 10am agenda attached',
    'Please review the quarterly report by Friday',
    'Your invoice for this month is attached',
    'Team lunch on Thursday let me know if you can join',
    'The project deadline has been extended to next week',
    'Annual leave request approved by HR department',
    'Training session scheduled for Monday at 2pm',
    'Please update your timesheet by end of day',
    'Code review comments added to your pull request',
]
labels = ['spam','spam','spam','spam','spam','spam',
          'ham','ham','ham','ham','ham','ham','ham','ham','ham']

print(f'Total emails: {len(spam_emails)}')
print(f'Spam: {labels.count("spam")} | Ham: {labels.count("ham")}')

In [ ]:
# TF-IDF + Logistic Regression pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import numpy as np

text_clf = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, stop_words='english')),
    ('clf',   LogisticRegression(random_state=42, max_iter=200))
])

# Cross-validation (small dataset, so use CV instead of split)
scores = cross_val_score(text_clf, spam_emails, labels, cv=3, scoring='accuracy')
print('Cross-validation accuracy:', scores.round(3))
print('Mean CV accuracy:', scores.mean().round(3))

# Fit on all data and predict new emails
text_clf.fit(spam_emails, labels)
new_emails = [
    'Click here to claim your prize money free gift',
    'Budget report for Q3 review attached',
]
predictions = text_clf.predict(new_emails)
for email, pred in zip(new_emails, predictions):
    print(f'\n["{email[:40]}..."] => {pred.upper()}')

## 7. Practical Lab 4 – Disease Detection (Classification)

### Objective
Build a synthetic disease risk classifier with multiple features.

### Explanation
Medical classification uses multiple risk factors. We simulate a realistic scenario — in production, always use real consented medical data.

In [ ]:
# Lab 4: Synthetic disease detection dataset
np.random.seed(42)
n = 500
df_d = pd.DataFrame({
    'Age':        np.random.randint(20, 80, n),
    'BMI':        np.random.uniform(18, 40, n),
    'Blood_Pres': np.random.randint(80, 160, n),
    'Glucose':    np.random.randint(70, 200, n),
    'Smoker':     np.random.randint(0, 2, n),
    'Exercise':   np.random.uniform(0, 10, n),
})
# Risk formula
risk = (0.03*df_d['Age'] + 0.04*df_d['BMI'] + 0.01*df_d['Blood_Pres'] +
        0.02*df_d['Glucose'] + 0.5*df_d['Smoker'] - 0.1*df_d['Exercise'])
p = 1 / (1 + np.exp(-(risk - 7)))
df_d['Disease'] = (np.random.rand(n) < p).astype(int)
print('Disease dataset:')
print(df_d.head())
print(f'Positive cases: {df_d["Disease"].sum()} / {n}')

In [ ]:
# Train and evaluate classifier
Xd = df_d[['Age','BMI','Blood_Pres','Glucose','Smoker','Exercise']]
yd = df_d['Disease']
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.2, stratify=yd, random_state=42)

disease_clf = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(random_state=42))])
disease_clf.fit(Xd_tr, yd_tr)
yd_pred = disease_clf.predict(Xd_te)

print('Disease Detection Results:')
print(f'Accuracy: {accuracy_score(yd_te, yd_pred):.4f}')
print(classification_report(yd_te, yd_pred, target_names=['No Disease','Disease']))
print('\n⚕️ In medical contexts, recall (sensitivity) is critical')
print('   to minimize missed diagnoses (false negatives).')

## 8. Practical Lab 5 – Model Comparison: Classification vs Regression Metrics

### Objective
Understand when each metric is appropriate and compare models side by side.

### Explanation
Choosing the wrong metric gives misleading results. Always match the metric to the problem type and business objective.

In [ ]:
# Lab 5: Metric comparison dashboard
print('=== CLASSIFICATION METRICS (Student Pass/Fail) ===')
from sklearn.metrics import precision_score, recall_score, f1_score
print(f'Accuracy:  {accuracy_score(y_te, y_pred):.4f}')
print(f'Precision: {precision_score(y_te, y_pred, pos_label="Pass"):.4f}')
print(f'Recall:    {recall_score(y_te, y_pred, pos_label="Pass"):.4f}')
print(f'F1-Score:  {f1_score(y_te, y_pred, pos_label="Pass"):.4f}')

print('\n=== REGRESSION METRICS (House Price) ===')
print(f'MAE:  {mean_absolute_error(yh_te, yh_pred):>12,.0f}')
print(f'RMSE: {mean_squared_error(yh_te, yh_pred)**0.5:>12,.0f}')
print(f'R²:   {r2_score(yh_te, yh_pred):>12.4f}')

print('\n=== METRIC SELECTION GUIDE ===')
print('Accuracy   → Balanced classes, equal cost of errors')
print('Precision  → False positives are expensive (e.g., spam filter)')
print('Recall     → False negatives are dangerous (e.g., disease detection)')
print('F1-Score   → Imbalanced classes, need balance of P & R')
print('R²         → Regression: how much variance is explained')
print('RMSE       → Regression: penalizes large errors more')

## 9. Exercises

**Exercise 1:** Modify the student dataset to include a `Homework_Avg` feature. Retrain and compare accuracy.

In [ ]:
# Exercise 1
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
np.random.seed(42)
# Generate dataset with Homework_Avg feature and compare accuracy

**Exercise 2:** On the housing dataset, compare LinearRegression vs Ridge (alpha=0.1, 1.0, 10.0). Which R² is highest?

In [ ]:
# Exercise 2
from sklearn.linear_model import LinearRegression, Ridge
# Compare Linear and Ridge at different alpha values

**Exercise 3:** Add 5 more spam and 5 more ham emails to the dataset. Retrain and test with 2 new sample emails.

In [ ]:
# Exercise 3 — Add more emails and retrain spam filter

## 10. Challenge Task – Salary Prediction

Generate a synthetic HR salary dataset (role, experience, education, skills count). Predict annual salary:
1. Generate data (200 employees)
2. Encode categorical columns
3. Train Ridge Regression
4. Print MAE, RMSE, R²
5. Predict salary for a new employee profile
6. Plot actual vs predicted

In [ ]:
# Challenge Task: Salary Prediction
np.random.seed(0)
n = 200
roles = ['Junior','Mid','Senior','Manager','Director']
role_sal = {'Junior':30000,'Mid':50000,'Senior':75000,'Manager':95000,'Director':120000}
edu = ['Diploma','Bachelor','Master','PhD']
edu_bonus = {'Diploma':0,'Bachelor':5000,'Master':15000,'PhD':25000}

df_sal = pd.DataFrame({
    'Experience': np.random.randint(0, 20, n),
    'Role':       np.random.choice(roles, n),
    'Education':  np.random.choice(edu, n),
    'Skills':     np.random.randint(3, 15, n),
})
df_sal['Salary'] = (
    df_sal['Role'].map(role_sal) +
    df_sal['Education'].map(edu_bonus) +
    1500 * df_sal['Experience'] +
    800 * df_sal['Skills'] +
    np.random.randn(n) * 3000
)

# One-hot encode
df_enc = pd.get_dummies(df_sal, columns=['Role','Education'], drop_first=True)
Xs = df_enc.drop('Salary', axis=1)
ys = df_enc['Salary']

Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(Xs, ys, test_size=0.2, random_state=0)
reg = Pipeline([('sc', StandardScaler()), ('ridge', Ridge(alpha=1.0))])
reg.fit(Xs_tr, ys_tr)
ys_pred = reg.predict(Xs_te)

print(f'Salary Prediction:')
print(f'  MAE:  {mean_absolute_error(ys_te, ys_pred):,.0f}')
print(f'  RMSE: {mean_squared_error(ys_te, ys_pred)**0.5:,.0f}')
print(f'  R²:   {r2_score(ys_te, ys_pred):.4f}')

## 11. Real-World AI/ML Application

| Domain | Classification Task | Regression Task |
|---|---|---|
| **Finance** | Fraud detection | Stock price prediction |
| **Healthcare** | Disease screening | Drug dosage optimization |
| **Education** | Pass/fail prediction | Grade prediction |
| **HR** | Attrition risk | Salary estimation |
| **Marketing** | Customer churn | Revenue forecasting |

## 12. Day Summary

| Topic | Key Point |
|---|---|
| Classification | Predict categories → Accuracy, F1, Precision, Recall |
| Regression | Predict numbers → MAE, RMSE, R² |
| TF-IDF | Convert text to ML-ready features |
| Ridge Regression | L2 regularization prevents overfitting |
| Metric selection | Always match metric to business objective |

**→ Day 6: Unsupervised Learning**